# The Proper EDA: Lending Club Loan Data (2007-2018)

A deep exploratory analysis of 2.26 million consumer loans originated through Lending Club. This notebook treats EDA as the product - not a quick pass before modeling, but a rigorous, structured investigation of data quality, distributional properties, temporal dynamics, multivariate structure, and portfolio-level credit analytics.

**Dataset:** [Lending Club Loan Data](https://www.kaggle.com/datasets/wordsforthewise/lending-club) - 2.26M loans, 151 features, 2007-2018.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, leaves_list
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
# -- Color Palette --
C = {
    "primary":    "#1e293b",
    "secondary":  "#334155",
    "accent":     "#2563eb",
    "accent2":    "#7c3aed",
    "risk":       "#dc2626",
    "safe":       "#059669",
    "warn":       "#d97706",
    "light_gray": "#f1f5f9",
    "mid_gray":   "#94a3b8",
    "dark_gray":  "#475569",
    "bg":         "#ffffff",
}

# Grade color scale (A=safest green through G=riskiest red)
GRADE_COLORS = {
    "A": "#059669", "B": "#10b981", "C": "#d97706",
    "D": "#f59e0b", "E": "#ef4444", "F": "#dc2626", "G": "#991b1b",
}

# Sequential blue palette for heatmaps
BLUE_CMAP = LinearSegmentedColormap.from_list("blue_seq", ["#f0f9ff", "#2563eb", "#1e3a5f"])
# Diverging palette
DIV_CMAP = LinearSegmentedColormap.from_list("div", ["#059669", "#f1f5f9", "#dc2626"])

# -- Matplotlib Style --
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#e2e8f0",
    "axes.labelcolor": C["primary"],
    "axes.titlecolor": C["primary"],
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.color": "#f1f5f9",
    "grid.linewidth": 0.8,
    "xtick.color": C["dark_gray"],
    "ytick.color": C["dark_gray"],
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial"],
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "legend.frameon": False,
    "legend.fontsize": 9,
})

def style_axis(ax, title="", subtitle="", xlabel="", ylabel=""):
    """Apply consistent styling to a matplotlib axis."""
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", color=C["primary"], pad=12, loc="left")
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10,
                color=C["dark_gray"], style="italic", va="bottom")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11, color=C["secondary"])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, color=C["secondary"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e2e8f0")
    ax.spines["bottom"].set_color("#e2e8f0")
    return ax

def fmt_pct(x, _=None):
    return f"{x:.0%}" if abs(x) < 10 else f"{x:.1%}"

def fmt_k(x, _=None):
    if abs(x) >= 1e6:
        return f"{x/1e6:.1f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

print("Style configured.")

In [ ]:
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv.gz", low_memory=False)

# Parse dates
df["issue_d"] = pd.to_datetime(df["issue_d"], format="mixed")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="mixed", errors="coerce")

# Derive key columns upfront
df["issue_year"] = df["issue_d"].dt.year
df["issue_month"] = df["issue_d"].dt.to_period("M")
df["credit_history_years"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25

# Binary default indicator (Charged Off or Default = 1, else 0)
default_statuses = ["Charged Off", "Default"]
df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)

print(f"Loaded: {df.shape[0]:,} loans x {df.shape[1]} features")
print(f"Date range: {df['issue_d'].min():%Y-%m} to {df['issue_d'].max():%Y-%m}")
print(f"Default rate: {df['is_default'].mean():.2%}")

---
## 1. Data Audit

Before any analysis, understand what you have: types, cardinality, memory footprint, duplicates, and structural organization of the 151 features.

In [ ]:
mem_usage = df.memory_usage(deep=True)
total_mb = mem_usage.sum() / 1e6

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total memory: {total_mb:,.1f} MB")
print(f"\nDtype counts:")
print(df.dtypes.value_counts().to_string())

In [ ]:
col_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.count(),
    "null_pct": df.isnull().mean(),
    "n_unique": df.nunique(),
    "pct_unique": df.nunique() / len(df),
    "top_value": df.apply(lambda x: x.value_counts().index[0] if x.count() > 0 else None),
    "top_freq_pct": df.apply(lambda x: x.value_counts(normalize=True).iloc[0] if x.count() > 0 else None),
    "memory_mb": df.memory_usage(deep=True)[1:] / 1e6,
})
col_summary = col_summary.sort_values("memory_mb", ascending=False)

print("Top 30 columns by memory usage:")
print(col_summary.head(30).to_string())

In [ ]:
obj_cols = df.select_dtypes(include="object").columns
dtype_issues = []
for col in obj_cols:
    sample = df[col].dropna().head(1000)
    numeric_frac = pd.to_numeric(sample.str.replace("[%,$]", "", regex=True), errors="coerce").notna().mean()
    if numeric_frac > 0.8:
        dtype_issues.append({"column": col, "numeric_fraction": numeric_frac, "sample_values": list(sample.head(3))})

if dtype_issues:
    print("Columns stored as object but appear numeric:")
    for d in dtype_issues:
        print(f"  {d['column']}: {d['numeric_fraction']:.0%} numeric | samples: {d['sample_values']}")
else:
    print("No dtype mismatches found.")

In [ ]:
card = col_summary[["n_unique", "pct_unique", "top_freq_pct"]].copy()
card = card.sort_values("n_unique", ascending=True).tail(40)

colors_bar = []
for idx in card.index:
    if card.loc[idx, "n_unique"] <= 2 or card.loc[idx, "top_freq_pct"] > 0.99:
        colors_bar.append(C["risk"])
    elif card.loc[idx, "n_unique"] > 1000:
        colors_bar.append(C["warn"])
    else:
        colors_bar.append(C["accent"])

fig, ax = plt.subplots(figsize=(12, 14))
ax.barh(range(len(card)), card["n_unique"], color=colors_bar, height=0.7)
ax.set_yticks(range(len(card)))
ax.set_yticklabels(card.index, fontsize=8)
ax.set_xscale("log")
style_axis(ax, title="Feature Cardinality (Top 40)",
           subtitle="Red = near-constant, Amber = high cardinality (>1000), Blue = normal",
           xlabel="Unique Values (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
near_const = col_summary[(col_summary["n_unique"] <= 2) | (col_summary["top_freq_pct"] > 0.99)]
if len(near_const) > 0:
    print(f"Near-constant columns ({len(near_const)}):")
    print(near_const[["n_unique", "top_value", "top_freq_pct"]].to_string())
else:
    print("No near-constant columns found.")

In [ ]:
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes:,} ({exact_dupes/len(df):.4%})")

near_dupe_cols = ["loan_amnt", "funded_amnt", "int_rate", "grade", "issue_d", "addr_state"]
available = [c for c in near_dupe_cols if c in df.columns]
near_dupes = df.duplicated(subset=available).sum()
print(f"Near-duplicate rows (same {', '.join(available)}): {near_dupes:,} ({near_dupes/len(df):.2%})")

In [ ]:
COLUMN_GROUPS = {
    "Loan Terms": ["id", "member_id", "loan_amnt", "funded_amnt", "funded_amnt_inv", "term", "int_rate",
                    "installment", "grade", "sub_grade", "url", "desc"],
    "Borrower Demographics": ["emp_title", "emp_length", "home_ownership", "annual_inc",
                               "verification_status", "addr_state", "zip_code",
                               "annual_inc_joint", "dti_joint", "verification_status_joint"],
    "Credit History": ["dti", "delinq_2yrs", "earliest_cr_line", "fico_range_low",
                       "fico_range_high", "inq_last_6mths", "open_acc", "pub_rec",
                       "revol_bal", "revol_util", "total_acc", "pub_rec_bankruptcies",
                       "mort_acc", "num_actv_bc_tl", "num_bc_sats", "num_bc_tl",
                       "num_il_tl", "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0",
                       "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd",
                       "num_tl_90g_dpd_24m", "num_tl_op_past_12m",
                       "pct_tl_nvr_dlq", "percent_bc_gt_75",
                       "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit",
                       "total_il_high_credit_limit", "acc_open_past_24mths",
                       "avg_cur_bal", "bc_open_to_buy", "bc_util",
                       "chargeoff_within_12_mths", "collections_12_mths_ex_med",
                       "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op",
                       "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl",
                       "mths_since_last_delinq", "mths_since_last_major_derog",
                       "mths_since_last_record", "mths_since_recent_bc",
                       "mths_since_recent_bc_dlq", "mths_since_recent_inq",
                       "mths_since_recent_revol_delinq", "tax_liens",
                       "tot_coll_amt", "tot_cur_bal", "total_rev_hi_lim",
                       "last_fico_range_high", "last_fico_range_low",
                       "acc_now_delinq", "delinq_amnt",
                       "num_accts_ever_120_pd", "num_actv_rev_tl",
                       "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m",
                       "mths_since_rcnt_il", "total_bal_il", "il_util",
                       "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util",
                       "inq_fi", "total_cu_tl", "inq_last_12m"],
    "Loan Status & Payments": ["loan_status", "pymnt_plan", "issue_d", "purpose", "title",
                                "initial_list_status", "application_type", "disbursement_method",
                                "out_prncp", "out_prncp_inv", "total_pymnt",
                                "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
                                "total_rec_late_fee", "recoveries",
                                "collection_recovery_fee", "last_pymnt_d",
                                "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
                                "policy_code"],
    "Hardship & Settlement": ["hardship_flag", "hardship_type", "hardship_reason",
                               "hardship_status", "hardship_amount",
                               "hardship_start_date", "hardship_end_date",
                               "hardship_length", "hardship_dpd",
                               "hardship_loan_status", "hardship_payoff_balance_amount",
                               "hardship_last_payment_amount",
                               "deferral_term", "payment_plan_start_date",
                               "orig_projected_additional_accrued_interest",
                               "debt_settlement_flag", "debt_settlement_flag_date",
                               "settlement_status", "settlement_date",
                               "settlement_amount", "settlement_percentage",
                               "settlement_term"],
    "Secondary Applicant": ["revol_bal_joint",
                             "sec_app_fico_range_low", "sec_app_fico_range_high",
                             "sec_app_earliest_cr_line", "sec_app_inq_last_6mths",
                             "sec_app_mort_acc", "sec_app_open_acc",
                             "sec_app_revol_util", "sec_app_open_act_il",
                             "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths",
                             "sec_app_collections_12_mths_ex_med",
                             "sec_app_mths_since_last_major_derog"],
}

for group, cols in COLUMN_GROUPS.items():
    present = [c for c in cols if c in df.columns]
    missing_from_df = [c for c in cols if c not in df.columns]
    print(f"\n{group}: {len(present)} features present" + (f", {len(missing_from_df)} not in data" if missing_from_df else ""))

uncategorized = set(df.columns) - set(c for cols in COLUMN_GROUPS.values() for c in cols)
derived = {"issue_year", "issue_month", "credit_history_years", "is_default"}
uncategorized -= derived
if uncategorized:
    print(f"\nUncategorized ({len(uncategorized)}): {sorted(uncategorized)}")

In [ ]:
print("Memory optimization recommendations:\n")
opt_savings = 0
for col in df.columns:
    if df[col].dtype == "float64":
        opt_savings += df[col].memory_usage(deep=True) * 0.5
    if df[col].dtype == "object" and df[col].nunique() < 50:
        current = df[col].memory_usage(deep=True)
        as_cat = df[col].astype("category").memory_usage(deep=True)
        opt_savings += current - as_cat

print(f"Estimated memory savings from downcasting: ~{opt_savings/1e6:,.0f} MB")
print(f"Current total: {df.memory_usage(deep=True).sum()/1e6:,.0f} MB")
print(f"Potential total: {(df.memory_usage(deep=True).sum() - opt_savings)/1e6:,.0f} MB")

---
## 2. Missing Data Analysis

Missingness is data. Whether a field is missing, and which other fields are missing at the same time, carries information about borrower type, application era, and risk. This section treats missing values as a structured signal, not a nuisance to impute away.

In [ ]:
miss_pct = df.isnull().mean().sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]

colors_miss = [C["risk"] if v > 0.5 else C["warn"] if v > 0.1 else C["accent"] for v in miss_pct.values]

fig, ax = plt.subplots(figsize=(12, max(10, len(miss_pct) * 0.25)))
ax.barh(range(len(miss_pct)), miss_pct.values, color=colors_miss, height=0.7)
ax.set_yticks(range(len(miss_pct)))
ax.set_yticklabels(miss_pct.index, fontsize=7)
ax.axvline(x=0.1, color=C["warn"], linestyle="--", lw=1, alpha=0.7, label="10%")
ax.axvline(x=0.5, color=C["risk"], linestyle="--", lw=1, alpha=0.7, label="50%")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.invert_yaxis()
style_axis(ax, title="Missing Data Rates by Feature",
           subtitle=f"{len(miss_pct)} of {len(df.columns)} columns have missing values",
           xlabel="% Missing")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

tier_counts = {
    "Mostly missing (>50%)": (miss_pct > 0.5).sum(),
    "Partially missing (10-50%)": ((miss_pct > 0.1) & (miss_pct <= 0.5)).sum(),
    "Mostly complete (<10%)": (miss_pct <= 0.1).sum(),
    "Fully complete": (df.isnull().mean() == 0).sum(),
}
print("\nMissingness tiers:")
for tier, count in tier_counts.items():
    print(f"  {tier}: {count} columns")

In [ ]:
top_missing = miss_pct.head(40).index.tolist()
temporal_miss = df.groupby("issue_year")[top_missing].apply(lambda x: x.isnull().mean())

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(temporal_miss.T, cmap=BLUE_CMAP, ax=ax, vmin=0, vmax=1,
            linewidths=0.3, linecolor="#e2e8f0",
            cbar_kws={"label": "% Missing", "format": mticker.PercentFormatter(1.0)})
style_axis(ax, title="Missingness by Origination Year",
           subtitle="Reveals when Lending Club added or removed fields from the application",
           xlabel="Origination Year", ylabel="")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
plt.tight_layout()
plt.show()

The heatmap above reveals Lending Club's evolving data schema. Key observations:
- **Joint application fields** (sec_app_*) appear only from 2017 onward, when Lending Club began offering joint loans
- **Hardship and settlement fields** are concentrated in 2018-2019, when these programs were introduced
- **Credit bureau fields** (mths_since_*, num_*) were added in bulk around 2012-2013 as Lending Club enriched their credit data pulls
- **Legacy fields** that were discontinued show high missingness in later years

This is not random missingness - it is structural, driven by product evolution.

In [ ]:
top30_miss = miss_pct.head(30).index.tolist()
miss_binary = df[top30_miss].isnull().astype(int)

co_miss = pd.DataFrame(index=top30_miss, columns=top30_miss, dtype=float)
for i, c1 in enumerate(top30_miss):
    for j, c2 in enumerate(top30_miss):
        if i <= j:
            both = (miss_binary[c1] & miss_binary[c2]).mean()
            max_single = max(miss_binary[c1].mean(), miss_binary[c2].mean())
            co_miss.loc[c1, c2] = both / max_single if max_single > 0 else 0
            co_miss.loc[c2, c1] = co_miss.loc[c1, c2]

# Cluster
dist_matrix = 1 - co_miss.values.astype(float)
np.fill_diagonal(dist_matrix, 0)
dist_matrix = np.maximum(dist_matrix, 0)
linkage_matrix = linkage(squareform(dist_matrix), method="ward")
order = leaves_list(linkage_matrix)
co_miss_ordered = co_miss.iloc[order, order]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(co_miss_ordered.astype(float), cmap=BLUE_CMAP, ax=ax, vmin=0, vmax=1,
            linewidths=0.3, linecolor="#e2e8f0", square=True,
            cbar_kws={"label": "Co-missingness Rate"})
style_axis(ax, title="Co-Missingness Matrix (Clustered)",
           subtitle="Which columns go missing together? Blocks reveal structural groups.",
           xlabel="", ylabel="")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=7, rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
plt.tight_layout()
plt.show()

The clustered co-missingness matrix reveals distinct structural groups:

1. **Joint application cluster** - All `sec_app_*` fields go missing together (>95% co-missingness). These fields are only populated for joint applications, which are a small fraction of loans.
2. **Hardship/settlement cluster** - Hardship and settlement fields co-occur because they relate to the same post-origination workout programs.
3. **Credit bureau enrichment cluster** - Fields like `mths_since_recent_bc`, `num_tl_op_past_12m`, etc. were all added in the same schema update. They are present or absent as a block.

These are not independent missing values - they are schema-driven groups. Any imputation strategy must respect these structures.

In [ ]:
miss_vs_default = []
target_cols = miss_pct[(miss_pct > 0.05) & (miss_pct < 0.80)].index

for col in target_cols:
    present_default = df.loc[df[col].notna(), "is_default"].mean()
    missing_default = df.loc[df[col].isna(), "is_default"].mean()
    delta = missing_default - present_default
    miss_vs_default.append({
        "column": col,
        "default_when_present": present_default,
        "default_when_missing": missing_default,
        "delta": delta,
    })

mvd = pd.DataFrame(miss_vs_default).sort_values("delta", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(8, len(mvd) * 0.3)))
colors_delta = [C["risk"] if d > 0 else C["safe"] for d in mvd["delta"]]
ax.barh(range(len(mvd)), mvd["delta"], color=colors_delta, height=0.7)
ax.set_yticks(range(len(mvd)))
ax.set_yticklabels(mvd["column"], fontsize=8)
ax.axvline(x=0, color=C["primary"], lw=1)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
style_axis(ax, title="Does Missingness Predict Default?",
           subtitle="Difference in default rate: missing minus present. Red = higher default when missing.",
           xlabel="Default Rate Delta")
plt.tight_layout()
plt.show()

In [ ]:
miss_sample = df[top30_miss].sample(n=min(50000, len(df)), random_state=42).isnull().astype(int).T
link = linkage(miss_sample, method="ward")

fig, ax = plt.subplots(figsize=(14, 8))
dendrogram(link, labels=top30_miss, leaf_rotation=90, leaf_font_size=8,
           color_threshold=0.7 * max(link[:, 2]), ax=ax)
style_axis(ax, title="Missingness Pattern Dendrogram",
           subtitle="Columns that cluster together share the same missingness structure",
           ylabel="Distance")
plt.tight_layout()
plt.show()

---
## 3. Univariate Deep Dives

### 3a. Continuous Features

Going beyond histograms: distribution fitting, QQ plots, robust statistics, and outlier quantification for every key continuous variable.

In [ ]:
from scipy.stats import trim_mean, median_abs_deviation

key_continuous = ["loan_amnt", "int_rate", "annual_inc", "dti", "revol_bal",
                  "revol_util", "total_acc", "open_acc", "installment",
                  "funded_amnt", "total_pymnt", "total_rec_int"]
key_continuous = [c for c in key_continuous if c in df.columns]

robust_stats = []
for col in key_continuous:
    s = df[col].dropna()
    if len(s) == 0:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    outlier_pct = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()

    robust_stats.append({
        "feature": col,
        "mean": s.mean(),
        "median": s.median(),
        "trimmed_mean_5pct": trim_mean(s, 0.05),
        "MAD": median_abs_deviation(s),
        "IQR": iqr,
        "skewness": s.skew(),
        "kurtosis": s.kurtosis(),
        "outlier_pct": outlier_pct,
    })

robust_df = pd.DataFrame(robust_stats).set_index("feature")
print("Robust Statistics Summary:")
print(robust_df.round(2).to_string())
print("\nFlags:")
for idx, row in robust_df.iterrows():
    flags = []
    if abs(row["skewness"]) > 2:
        flags.append(f"highly skewed ({row['skewness']:.1f})")
    if row["outlier_pct"] > 0.05:
        flags.append(f"heavy outliers ({row['outlier_pct']:.1%})")
    if abs(row["mean"] - row["median"]) / (row["MAD"] + 1e-9) > 1:
        flags.append("mean/median divergence")
    if flags:
        print(f"  {idx}: {', '.join(flags)}")

In [ ]:
from scipy.stats import norm, lognorm, kstest, shapiro

def distribution_deep_dive(series, name, df_full=None, target_col="is_default", figsize=(16, 10)):
    """Full distributional analysis panel for a continuous feature."""
    s = series.dropna()
    s_clipped = s.clip(s.quantile(0.001), s.quantile(0.999))

    fig, axes = plt.subplots(2, 2, figsize=figsize)

    # Panel 1: Histogram + KDE + Fitted Distributions
    ax = axes[0, 0]
    ax.hist(s_clipped, bins=80, density=True, alpha=0.3, color=C["accent"], edgecolor="none")
    kde_x = np.linspace(s_clipped.min(), s_clipped.max(), 500)

    mu, sigma = norm.fit(s_clipped)
    ks_norm = kstest(s_clipped.sample(min(5000, len(s_clipped)), random_state=42), "norm", args=(mu, sigma)).statistic
    ax.plot(kde_x, norm.pdf(kde_x, mu, sigma), color=C["risk"], lw=1.5, ls="--", label=f"Normal (KS={ks_norm:.3f})")

    if (s_clipped > 0).all():
        shape_ln, loc_ln, scale_ln = lognorm.fit(s_clipped, floc=0)
        ks_ln = kstest(s_clipped.sample(min(5000, len(s_clipped)), random_state=42), "lognorm", args=(shape_ln, loc_ln, scale_ln)).statistic
        ax.plot(kde_x, lognorm.pdf(kde_x, shape_ln, loc_ln, scale_ln), color=C["safe"], lw=1.5, ls="-.", label=f"Lognormal (KS={ks_ln:.3f})")

    ax.legend(fontsize=8)
    style_axis(ax, title=f"{name}: Distribution Fitting", xlabel=name, ylabel="Density")

    # Panel 2: QQ Plot
    ax = axes[0, 1]
    qq_sample = s_clipped.sample(min(5000, len(s_clipped)), random_state=42)
    qq = stats.probplot(qq_sample, dist="norm")
    ax.scatter(qq[0][0], qq[0][1], s=4, alpha=0.3, color=C["accent"])
    ax.plot(qq[0][0], qq[1][0] * qq[0][0] + qq[1][1], color=C["risk"], lw=1.5)
    style_axis(ax, title=f"{name}: Normal QQ Plot", xlabel="Theoretical Quantiles", ylabel="Sample Quantiles")

    # Panel 3: Box Plot with Outlier Thresholds
    ax = axes[1, 0]
    bp = ax.boxplot(s.values, vert=False, widths=0.6, patch_artist=True,
                     boxprops=dict(facecolor=C["accent"], alpha=0.3),
                     medianprops=dict(color=C["risk"], lw=2),
                     flierprops=dict(marker=".", markersize=1, alpha=0.1))
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    ax.axvline(q1 - 1.5 * iqr, color=C["warn"], ls="--", lw=1, label=f"1.5x IQR: {q1-1.5*iqr:,.0f}")
    ax.axvline(q3 + 1.5 * iqr, color=C["warn"], ls="--", lw=1, label=f"1.5x IQR: {q3+1.5*iqr:,.0f}")
    outlier_pct = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()
    ax.legend(fontsize=7, loc="upper right")
    style_axis(ax, title=f"{name}: Outlier Analysis ({outlier_pct:.1%} outliers)", xlabel=name)

    # Panel 4: Violin by Default Status
    ax = axes[1, 1]
    if df_full is not None and target_col in df_full.columns:
        mask = df_full[name].notna()
        data_0 = df_full.loc[mask & (df_full[target_col] == 0), name]
        data_1 = df_full.loc[mask & (df_full[target_col] == 1), name]
        # Clip for visualization
        clip_low, clip_high = s.quantile(0.001), s.quantile(0.999)
        data_0_clip = data_0.clip(clip_low, clip_high)
        data_1_clip = data_1.clip(clip_low, clip_high)
        parts = ax.violinplot([data_0_clip.values, data_1_clip.values], positions=[0, 1], showmedians=True)
        for i, pc in enumerate(parts["bodies"]):
            pc.set_facecolor(C["safe"] if i == 0 else C["risk"])
            pc.set_alpha(0.4)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Performing", "Default"])
        style_axis(ax, title=f"{name}: Distribution by Outcome", ylabel=name)

    fig.suptitle(f"Deep Dive: {name}", fontsize=16, fontweight="bold", color=C["primary"], y=1.01)
    plt.tight_layout()
    plt.show()

#### Loan Amount

In [ ]:
distribution_deep_dive(df["loan_amnt"], "loan_amnt", df_full=df)

#### Interest Rate

In [ ]:
distribution_deep_dive(df["int_rate"], "int_rate", df_full=df)

#### Annual Income

In [ ]:
distribution_deep_dive(df["annual_inc"], "annual_inc", df_full=df)

In [ ]:
# Log-transform check for annual_inc
s = df["annual_inc"].dropna()
s_pos = s[s > 0]
s_log = np.log1p(s_pos)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(s_pos.clip(0, s_pos.quantile(0.99)), bins=80, alpha=0.5, color=C["accent"], density=True, edgecolor="none")
style_axis(axes[0], title="annual_inc: Raw", xlabel="Annual Income")
axes[1].hist(s_log, bins=80, alpha=0.5, color=C["safe"], density=True, edgecolor="none")
_, p_val = shapiro(s_log.sample(min(5000, len(s_log)), random_state=42))
style_axis(axes[1], title=f"annual_inc: Log-Transformed (Shapiro p={p_val:.4f})", xlabel="log(1 + Annual Income)")
plt.tight_layout()
plt.show()

#### Debt-to-Income Ratio

In [ ]:
distribution_deep_dive(df["dti"], "dti", df_full=df)

#### Revolving Balance

In [ ]:
distribution_deep_dive(df["revol_bal"], "revol_bal", df_full=df)

In [ ]:
# Log-transform check for revol_bal
s = df["revol_bal"].dropna()
s_pos = s[s > 0]
s_log = np.log1p(s_pos)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(s_pos.clip(0, s_pos.quantile(0.99)), bins=80, alpha=0.5, color=C["accent"], density=True, edgecolor="none")
style_axis(axes[0], title="revol_bal: Raw", xlabel="Revolving Balance")
axes[1].hist(s_log, bins=80, alpha=0.5, color=C["safe"], density=True, edgecolor="none")
_, p_val = shapiro(s_log.sample(min(5000, len(s_log)), random_state=42))
style_axis(axes[1], title=f"revol_bal: Log-Transformed (Shapiro p={p_val:.4f})", xlabel="log(1 + Revolving Balance)")
plt.tight_layout()
plt.show()

#### Revolving Utilization

In [ ]:
distribution_deep_dive(df["revol_util"], "revol_util", df_full=df)

#### Monthly Installment

In [ ]:
distribution_deep_dive(df["installment"], "installment", df_full=df)

#### Total Accounts

In [ ]:
distribution_deep_dive(df["total_acc"], "total_acc", df_full=df)

#### Open Accounts

In [ ]:
distribution_deep_dive(df["open_acc"], "open_acc", df_full=df)

### 3b. Categorical Features

Frequency analysis, concentration metrics, and rare category detection for the key categorical dimensions.

In [ ]:
def categorical_deep_dive(series, name, df_full=None, target_col="is_default", figsize=(14, 5)):
    """Frequency analysis and concentration metrics for a categorical feature."""
    s = series.dropna()
    vc = s.value_counts()
    vc_pct = s.value_counts(normalize=True)

    n_cats = len(vc)
    hhi = (vc_pct ** 2).sum()
    rare = (vc_pct < 0.01).sum()

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # Panel 1: Frequency bar chart
    ax = axes[0]
    top_n = min(20, n_cats)
    plot_data = vc.head(top_n)
    ax.bar(range(len(plot_data)), plot_data.values, color=C["accent"], alpha=0.8)
    ax.set_xticks(range(len(plot_data)))
    ax.set_xticklabels(plot_data.index, rotation=45, ha="right", fontsize=8)
    for i, (v, pct) in enumerate(zip(plot_data.values, vc_pct.head(top_n).values)):
        ax.text(i, v, f"{pct:.1%}", ha="center", va="bottom", fontsize=7, color=C["dark_gray"])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
    style_axis(ax, title=f"{name}: Frequency Distribution",
               subtitle=f"{n_cats} categories | HHI={hhi:.3f} | {rare} rare (<1%)",
               ylabel="Count")

    # Panel 2: Default rate by category
    ax = axes[1]
    if df_full is not None and target_col in df_full.columns:
        default_by_cat = df_full.groupby(name)[target_col].agg(["mean", "count"])
        default_by_cat = default_by_cat[default_by_cat["count"] >= 100].sort_values("mean", ascending=False)
        plot_cats = default_by_cat.head(top_n)
        bar_colors = [C["risk"] if r > df_full[target_col].mean() else C["safe"] for r in plot_cats["mean"]]
        ax.barh(range(len(plot_cats)), plot_cats["mean"], color=bar_colors, height=0.7)
        ax.set_yticks(range(len(plot_cats)))
        ax.set_yticklabels(plot_cats.index, fontsize=8)
        ax.axvline(df_full[target_col].mean(), color=C["primary"], ls="--", lw=1, alpha=0.7,
                    label=f"Overall: {df_full[target_col].mean():.2%}")
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        ax.legend(fontsize=8)
        ax.invert_yaxis()
        style_axis(ax, title=f"{name}: Default Rate by Category", xlabel="Default Rate")

    plt.tight_layout()
    plt.show()

#### Grade

In [ ]:
categorical_deep_dive(df["grade"], "grade", df_full=df)

#### Sub Grade

In [ ]:
categorical_deep_dive(df["sub_grade"], "sub_grade", df_full=df)

#### Home Ownership

In [ ]:
categorical_deep_dive(df["home_ownership"], "home_ownership", df_full=df)

#### Purpose

In [ ]:
categorical_deep_dive(df["purpose"], "purpose", df_full=df)

#### Verification Status

In [ ]:
categorical_deep_dive(df["verification_status"], "verification_status", df_full=df)

#### Loan Status

In [ ]:
categorical_deep_dive(df["loan_status"], "loan_status", df_full=df)

#### Term

In [ ]:
categorical_deep_dive(df["term"], "term", df_full=df)

#### Application Type

In [ ]:
categorical_deep_dive(df["application_type"], "application_type", df_full=df)

In [ ]:
status_vc = df["loan_status"].value_counts()
fig, ax = plt.subplots(figsize=(12, 5))
colors_status = [C["safe"] if "Paid" in s else C["risk"] if s in ["Charged Off", "Default"]
                 else C["warn"] for s in status_vc.index]
ax.barh(range(len(status_vc)), status_vc.values, color=colors_status, height=0.7)
ax.set_yticks(range(len(status_vc)))
ax.set_yticklabels(status_vc.index, fontsize=9)
for i, v in enumerate(status_vc.values):
    ax.text(v, i, f"  {v:,} ({v/len(df):.1%})", va="center", fontsize=8, color=C["dark_gray"])
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
style_axis(ax, title="Loan Status Distribution",
           subtitle="Green = resolved performing, Red = credit loss, Amber = in progress/late")
plt.tight_layout()
plt.show()

In [ ]:
emp_map = {"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
           "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10}
df["emp_length_num"] = df["emp_length"].map(emp_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency
ax = axes[0]
emp_vc = df["emp_length"].value_counts().reindex(emp_map.keys())
ax.bar(range(len(emp_vc)), emp_vc.values, color=C["accent"], alpha=0.8)
ax.set_xticks(range(len(emp_vc)))
ax.set_xticklabels(emp_vc.index, rotation=45, ha="right", fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
missing_pct = df["emp_length"].isna().mean()
style_axis(ax, title="Employment Length Distribution",
           subtitle=f"{missing_pct:.1%} missing", ylabel="Count")

# Default rate by emp_length
ax = axes[1]
dr_emp = df.groupby("emp_length_num")["is_default"].mean()
ax.bar(dr_emp.index, dr_emp.values, color=C["accent"], alpha=0.8)
ax.set_xticks(range(11))
ax.set_xticklabels(["<1"] + [str(i) for i in range(1, 10)] + ["10+"], fontsize=8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.axhline(df["is_default"].mean(), color=C["risk"], ls="--", lw=1, label=f"Overall: {df['is_default'].mean():.2%}")
ax.legend()
style_axis(ax, title="Default Rate by Employment Length", xlabel="Years", ylabel="Default Rate")

plt.tight_layout()
plt.show()

### 3c. Derived and Temporal Features

In [ ]:
monthly = df.groupby(df["issue_d"].dt.to_period("M")).agg(
    count=("loan_amnt", "count"),
    volume=("loan_amnt", "sum")
)
monthly.index = monthly.index.to_timestamp()

fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.fill_between(monthly.index, monthly["count"], alpha=0.3, color=C["accent"])
ax1.plot(monthly.index, monthly["count"], color=C["accent"], lw=1.5, label="Loan Count")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
style_axis(ax1, title="Lending Club Origination Volume Over Time",
           subtitle="Monthly loan count and dollar volume (2007-2018)",
           ylabel="Loan Count")

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["volume"] / 1e6, color=C["warn"], lw=2, ls="--", label="$ Volume (M)")
ax2.set_ylabel("Dollar Volume ($M)", color=C["warn"], fontsize=11)
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_color(C["warn"])

events = {"2008-09": "Lehman\nBrothers", "2014-12": "LC IPO", "2016-05": "CEO\nResigns"}
for date_str, label in events.items():
    x = pd.Timestamp(date_str)
    if monthly.index.min() <= x <= monthly.index.max():
        ax1.axvline(x, color=C["risk"], ls=":", lw=1, alpha=0.6)
        ax1.text(x, ax1.get_ylim()[1] * 0.95, label, ha="center", fontsize=8,
                 color=C["risk"], fontweight="bold")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
s = df["credit_history_years"].dropna()
ax.hist(s.clip(0, s.quantile(0.999)), bins=80, color=C["accent"], alpha=0.5, density=True, edgecolor="none")
ax.axvline(s.median(), color=C["risk"], lw=2, ls="--", label=f"Median: {s.median():.1f} years")
style_axis(ax, title="Credit History Length at Origination",
           subtitle=f"Computed from earliest_cr_line. Range: {s.min():.1f} to {s.max():.1f} years",
           xlabel="Years of Credit History", ylabel="Density")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Temporal Analysis

How did the lending population, risk characteristics, and outcomes evolve across Lending Club's 12-year history? This section tracks feature drift, underwriting changes, and vintage-level default patterns.

In [ ]:
drift_features = ["loan_amnt", "int_rate", "dti", "annual_inc"]
drift_features = [c for c in drift_features if c in df.columns]

for feat in drift_features:
    subset = df[["issue_year", feat]].dropna()
    q_low, q_high = subset[feat].quantile(0.01), subset[feat].quantile(0.99)
    subset = subset[(subset[feat] >= q_low) & (subset[feat] <= q_high)]

    years = sorted(subset["issue_year"].unique())
    fig, ax = plt.subplots(figsize=(14, max(6, len(years) * 0.5)))

    for i, year in enumerate(years):
        year_data = subset.loc[subset["issue_year"] == year, feat]
        kde_x = np.linspace(q_low, q_high, 300)
        try:
            kde = stats.gaussian_kde(year_data)
            kde_y = kde(kde_x)
            kde_y_scaled = kde_y / kde_y.max() * 0.8
            ax.fill_between(kde_x, i, i + kde_y_scaled, alpha=0.4, color=C["accent"])
            ax.plot(kde_x, i + kde_y_scaled, color=C["primary"], lw=0.8)
        except Exception:
            pass

    ax.set_yticks(range(len(years)))
    ax.set_yticklabels(years, fontsize=9)
    ax.invert_yaxis()
    style_axis(ax, title=f"Feature Drift: {feat}",
               subtitle=f"Distribution shift by origination year (2007-2018)",
               xlabel=feat, ylabel="Vintage Year")
    plt.tight_layout()
    plt.show()

In [ ]:
grade_order = ["A", "B", "C", "D", "E", "F", "G"]
grade_by_year = df.groupby(["issue_year", "grade"]).size().unstack(fill_value=0)
grade_by_year_pct = grade_by_year.div(grade_by_year.sum(axis=1), axis=0)
grade_by_year_pct = grade_by_year_pct[[c for c in grade_order if c in grade_by_year_pct.columns]]

fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(grade_by_year_pct.index, grade_by_year_pct.values.T,
             labels=grade_by_year_pct.columns,
             colors=[GRADE_COLORS.get(g, C["mid_gray"]) for g in grade_by_year_pct.columns],
             alpha=0.8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlim(grade_by_year_pct.index.min(), grade_by_year_pct.index.max())
ax.legend(loc="upper right", ncol=7, fontsize=9)
style_axis(ax, title="Grade Composition by Vintage Year",
           subtitle="Did Lending Club loosen or tighten underwriting over time?",
           xlabel="Origination Year", ylabel="% of Originations")
plt.tight_layout()
plt.show()

In [ ]:
vintage_defaults = df.groupby("issue_year").agg(
    total=("is_default", "count"),
    defaults=("is_default", "sum"),
).assign(default_rate=lambda x: x["defaults"] / x["total"])

# Compute approximate months on books for context
df["mob"] = ((pd.Timestamp("2019-01-01") - df["issue_d"]).dt.days / 30.44).astype(int).clip(0)
vintage_defaults["avg_mob"] = df.groupby("issue_year")["mob"].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
bar_colors = [C["risk"] if r > 0.20 else C["warn"] if r > 0.15 else C["safe"]
              for r in vintage_defaults["default_rate"]]
ax.bar(vintage_defaults.index, vintage_defaults["default_rate"], color=bar_colors, alpha=0.8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
for yr, row in vintage_defaults.iterrows():
    ax.text(yr, row["default_rate"], f"{row['default_rate']:.1%}", ha="center", va="bottom", fontsize=8)
style_axis(ax, title="Cumulative Default Rate by Vintage",
           subtitle="Earlier vintages have more seasoning (more time to default)",
           xlabel="Origination Year", ylabel="Default Rate")

ax = axes[1]
ax.bar(vintage_defaults.index, vintage_defaults["avg_mob"], color=C["accent"], alpha=0.8)
for yr, row in vintage_defaults.iterrows():
    ax.text(yr, row["avg_mob"], f"{row['avg_mob']:.0f}", ha="center", va="bottom", fontsize=8)
style_axis(ax, title="Average Months on Books by Vintage",
           subtitle="Recent vintages have less time to season - default rates will rise",
           xlabel="Origination Year", ylabel="Avg Months on Books")

plt.tight_layout()
plt.show()

In [ ]:
vintage_grade = df.groupby(["issue_year", "grade"])["is_default"].mean().unstack()
vintage_grade = vintage_grade[[c for c in grade_order if c in vintage_grade.columns]]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(vintage_grade, cmap=DIV_CMAP, annot=True, fmt=".1%", ax=ax,
            linewidths=0.5, linecolor="#e2e8f0", vmin=0,
            cbar_kws={"label": "Default Rate", "format": mticker.PercentFormatter(1.0)})
style_axis(ax, title="Default Rate Heatmap: Vintage x Grade",
           subtitle="Darker red = higher default rate. Columns = credit quality, rows = origination year.",
           xlabel="Grade", ylabel="Vintage Year")
plt.tight_layout()
plt.show()

In [ ]:
rate_by_grade_year = df.groupby(["issue_year", "grade"])["int_rate"].median().unstack()
rate_by_grade_year = rate_by_grade_year[[c for c in grade_order if c in rate_by_grade_year.columns]]

fig, ax = plt.subplots(figsize=(14, 6))
for g in rate_by_grade_year.columns:
    ax.plot(rate_by_grade_year.index, rate_by_grade_year[g], marker="o", markersize=4,
            lw=2, color=GRADE_COLORS.get(g, C["mid_gray"]), label=g)
ax.legend(title="Grade", ncol=7, fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
style_axis(ax, title="Median Interest Rate by Grade Over Time",
           subtitle="How did pricing evolve across credit tiers?",
           xlabel="Origination Year", ylabel="Median Interest Rate")
plt.tight_layout()
plt.show()

In [ ]:
pctiles = df.groupby("issue_year")["loan_amnt"].quantile([0.25, 0.50, 0.75, 0.95]).unstack()

fig, ax = plt.subplots(figsize=(14, 6))
for pct, ls in [(0.25, ":"), (0.50, "-"), (0.75, "--"), (0.95, "-.")]:
    ax.plot(pctiles.index, pctiles[pct], lw=2, ls=ls, color=C["accent"],
            label=f"{pct:.0%}ile", marker="o", markersize=4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=9)
ax.fill_between(pctiles.index, pctiles[0.25], pctiles[0.75], alpha=0.1, color=C["accent"])
style_axis(ax, title="Loan Amount Percentiles by Vintage",
           subtitle="IQR shaded. Loan sizes grew steadily across all percentiles.",
           xlabel="Origination Year", ylabel="Loan Amount ($)")
plt.tight_layout()
plt.show()

### Temporal Analysis Summary

Key findings:
- **Volume:** Lending Club grew exponentially from 2007 to 2015, plateauing around 2016 after the CEO resignation scandal. Monthly origination volume peaked at roughly $1B.
- **Grade drift:** The proportion of higher-risk grades (D, E, F, G) increased over time, suggesting gradual underwriting relaxation to fuel growth.
- **Interest rates:** Declined within each grade over time, partly tracking the broader rate environment and partly reflecting competitive pressure.
- **Seasoning bias:** Recent vintages show artificially low default rates because they have not had enough time to season. A 2018 vintage with 12 months of history will look better than a 2012 vintage with 72 months, purely due to time.
- **Vintage quality:** Controlling for seasoning, pre-2012 vintages had higher default rates, influenced by the financial crisis and early-stage underwriting.

---
## 5. Bivariate & Conditional Analysis

Move beyond univariate summaries. How do features relate to each other and to the default outcome?

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=["is_default", "issue_year", "mob"], errors="ignore").columns
# Use a sample for speed on correlation
corr_sample = df[numeric_cols].sample(min(100000, len(df)), random_state=42)
corr_spearman = corr_sample.corr(method="spearman")

pairs = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        pairs.append({
            "feature_1": numeric_cols[i],
            "feature_2": numeric_cols[j],
            "spearman_r": corr_spearman.iloc[i, j],
        })
pairs_df = pd.DataFrame(pairs).sort_values("spearman_r", ascending=False)

print("Top 20 positive correlations:")
print(pairs_df.head(20)[["feature_1", "feature_2", "spearman_r"]].to_string(index=False))
print("\nTop 20 negative correlations:")
print(pairs_df.tail(20)[["feature_1", "feature_2", "spearman_r"]].to_string(index=False))

In [ ]:
from scipy.stats import pointbiserialr

pb_corrs = []
sample_pb = df.sample(min(100000, len(df)), random_state=42)
for col in numeric_cols:
    s = sample_pb[[col, "is_default"]].dropna()
    if len(s) < 100:
        continue
    r, p = pointbiserialr(s["is_default"], s[col])
    pb_corrs.append({"feature": col, "r": r, "p_value": p, "abs_r": abs(r)})

pb_df = pd.DataFrame(pb_corrs).sort_values("abs_r", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(8, len(pb_df) * 0.25)))
colors_pb = [C["risk"] if r > 0 else C["safe"] for r in pb_df["r"]]
ax.barh(range(len(pb_df)), pb_df["r"], color=colors_pb, height=0.7)
ax.set_yticks(range(len(pb_df)))
ax.set_yticklabels(pb_df["feature"], fontsize=7)
ax.axvline(0, color=C["primary"], lw=1)
style_axis(ax, title="Feature Association with Default (Point-Biserial r)",
           subtitle="Red = positive association (higher value, more default). Green = protective.",
           xlabel="Point-Biserial Correlation")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
grade_order_present = [g for g in grade_order if g in df["grade"].unique()]
plot_data = [df.loc[(df["grade"] == g) & (df["annual_inc"] < df["annual_inc"].quantile(0.99)), "annual_inc"].values
             for g in grade_order_present]

parts = ax.violinplot(plot_data, positions=range(len(grade_order_present)), showmedians=True, showextrema=False)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(GRADE_COLORS.get(grade_order_present[i], C["mid_gray"]))
    pc.set_alpha(0.6)
ax.set_xticks(range(len(grade_order_present)))
ax.set_xticklabels(grade_order_present)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
style_axis(ax, title="Annual Income Distribution by Grade",
           subtitle="Higher-grade borrowers have higher incomes - but with significant overlap",
           xlabel="Grade", ylabel="Annual Income")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
dti_clean = df.loc[df["dti"].between(0, 60), ["dti", "is_default"]]
for val, label, color in [(0, "Performing", C["safe"]), (1, "Default", C["risk"])]:
    subset = dti_clean.loc[dti_clean["is_default"] == val, "dti"]
    ax.hist(subset, bins=60, density=True, alpha=0.3, color=color, label=label, edgecolor="none")
    kde = stats.gaussian_kde(subset)
    x = np.linspace(0, 60, 300)
    ax.plot(x, kde(x), color=color, lw=2)
style_axis(ax, title="DTI Distribution: Performing vs. Default",
           subtitle="Defaulted borrowers carried higher debt-to-income ratios at origination",
           xlabel="DTI (%)", ylabel="Density")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(contingency_table):
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    return np.sqrt(chi2 / (n * max(min_dim, 1)))

cat_pairs = [
    ("purpose", "is_default"),
    ("home_ownership", "is_default"),
    ("grade", "home_ownership"),
    ("verification_status", "is_default"),
    ("term", "is_default"),
]

chi2_results = []
for col1, col2 in cat_pairs:
    if col1 not in df.columns or col2 not in df.columns:
        continue
    ct = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, _ = chi2_contingency(ct)
    v = cramers_v(ct)
    chi2_results.append({"pair": f"{col1} x {col2}", "chi2": chi2, "p_value": p, "dof": dof, "cramers_v": v})

chi2_df = pd.DataFrame(chi2_results).sort_values("cramers_v", ascending=False)
print("Chi-Squared Tests of Association:")
print(chi2_df.to_string(index=False))

In [ ]:
interaction = df.groupby(["grade", "purpose"])["is_default"].agg(["mean", "count"]).reset_index()
interaction = interaction[interaction["count"] >= 50]
pivot = interaction.pivot(index="purpose", columns="grade", values="mean")
pivot = pivot[[c for c in grade_order if c in pivot.columns]]
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap=DIV_CMAP, annot=True, fmt=".1%", ax=ax,
            linewidths=0.5, linecolor="#e2e8f0", vmin=0,
            cbar_kws={"label": "Default Rate", "format": mticker.PercentFormatter(1.0)})
style_axis(ax, title="Default Rate: Grade x Loan Purpose",
           subtitle="Are certain purposes riskier within the same credit grade?",
           xlabel="Grade", ylabel="Purpose")
plt.tight_layout()
plt.show()

In [ ]:
state_stats = df.groupby("addr_state").agg(
    default_rate=("is_default", "mean"),
    loan_count=("loan_amnt", "count"),
    avg_loan=("loan_amnt", "mean"),
    avg_rate=("int_rate", "mean"),
).reset_index()

fig = px.choropleth(state_stats, locations="addr_state", locationmode="USA-states",
                    color="default_rate", scope="usa",
                    color_continuous_scale=["#059669", "#fef3c7", "#dc2626"],
                    hover_data=["loan_count", "avg_loan", "avg_rate"],
                    labels={"default_rate": "Default Rate", "addr_state": "State"})
fig.update_layout(
    title=dict(text="Default Rate by State", font=dict(size=18, color="#1e293b")),
    geo=dict(bgcolor="#ffffff", lakecolor="#f0f9ff"),
    paper_bgcolor="#ffffff",
    margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Income vs loan amount
ax = axes[0]
sample = df[["annual_inc", "loan_amnt", "grade"]].dropna().sample(50000, random_state=42)
for g in grade_order:
    mask = sample["grade"] == g
    if mask.sum() > 0:
        ax.scatter(sample.loc[mask, "annual_inc"], sample.loc[mask, "loan_amnt"],
                   s=3, alpha=0.15, color=GRADE_COLORS.get(g, C["mid_gray"]), label=g)
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend(title="Grade", markerscale=5, fontsize=8)
style_axis(ax, title="Income vs. Loan Amount (log-log)",
           xlabel="Annual Income ($)", ylabel="Loan Amount ($)")

# DTI vs int_rate
ax = axes[1]
sample2 = df[["dti", "int_rate", "is_default"]].dropna()
sample2 = sample2[sample2["dti"].between(0, 50)].sample(50000, random_state=42)
for val, label, color in [(0, "Performing", C["safe"]), (1, "Default", C["risk"])]:
    mask = sample2["is_default"] == val
    ax.scatter(sample2.loc[mask, "dti"], sample2.loc[mask, "int_rate"],
               s=3, alpha=0.1, color=color, label=label)
ax.legend(markerscale=5, fontsize=8)
style_axis(ax, title="DTI vs. Interest Rate by Outcome",
           xlabel="DTI (%)", ylabel="Interest Rate (%)")

plt.tight_layout()
plt.show()

---
## 6. Multivariate Structure

Pairwise analysis misses interactions. PCA, clustering, and dimensionality reduction reveal latent structure in the borrower population.

In [ ]:
pca_features = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti",
                "delinq_2yrs", "fico_range_low", "inq_last_6mths", "open_acc",
                "pub_rec", "revol_bal", "revol_util", "total_acc",
                "out_prncp", "total_pymnt", "total_rec_int", "total_rec_prncp"]
pca_features = [c for c in pca_features if c in df.columns]

pca_df = df[pca_features].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df)

pca = PCA(n_components=min(10, len(pca_features)))
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_,
       color=C["accent"], alpha=0.8)
ax.set_xticks(range(1, len(pca.explained_variance_ratio_) + 1))
style_axis(ax, title="Scree Plot", xlabel="Principal Component", ylabel="Explained Variance Ratio")

ax = axes[1]
cumvar = np.cumsum(pca.explained_variance_ratio_)
ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o", color=C["accent"], lw=2)
ax.axhline(0.8, ls="--", color=C["warn"], lw=1, label="80%")
ax.axhline(0.95, ls="--", color=C["risk"], lw=1, label="95%")
n_80 = np.argmax(cumvar >= 0.8) + 1
ax.axvline(n_80, ls=":", color=C["dark_gray"], lw=1)
ax.text(n_80 + 0.2, 0.5, f"{n_80} components\nfor 80%", fontsize=9, color=C["dark_gray"])
ax.legend()
ax.set_xticks(range(1, len(cumvar) + 1))
style_axis(ax, title="Cumulative Explained Variance", xlabel="Components", ylabel="Cumulative Variance")

plt.tight_layout()
plt.show()

In [ ]:
loadings = pd.DataFrame(pca.components_[:5].T, index=pca_features,
                         columns=[f"PC{i+1}" for i in range(5)])

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(loadings, cmap="RdBu_r", center=0, annot=True, fmt=".2f", ax=ax,
            linewidths=0.5, linecolor="#e2e8f0",
            cbar_kws={"label": "Loading"})
style_axis(ax, title="PCA Loadings: Top 5 Components",
           subtitle="What financial dimensions drive the most variance?",
           xlabel="Component", ylabel="Feature")
plt.tight_layout()
plt.show()

In [ ]:
pca_plot = pca_df.copy()
pca_plot["PC1"] = X_pca[:, 0]
pca_plot["PC2"] = X_pca[:, 1]
pca_plot["grade"] = df.loc[pca_df.index, "grade"]
pca_plot["is_default"] = df.loc[pca_df.index, "is_default"]

sample_pca = pca_plot.sample(min(30000, len(pca_plot)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for g in grade_order:
    mask = sample_pca["grade"] == g
    if mask.sum() > 0:
        ax.scatter(sample_pca.loc[mask, "PC1"], sample_pca.loc[mask, "PC2"],
                   s=3, alpha=0.15, color=GRADE_COLORS.get(g, C["mid_gray"]), label=g)
ax.legend(title="Grade", markerscale=5, fontsize=8)
style_axis(ax, title="PCA: PC1 vs PC2 (colored by Grade)", xlabel="PC1", ylabel="PC2")

ax = axes[1]
for val, label, color in [(0, "Performing", C["safe"]), (1, "Default", C["risk"])]:
    mask = sample_pca["is_default"] == val
    ax.scatter(sample_pca.loc[mask, "PC1"], sample_pca.loc[mask, "PC2"],
               s=3, alpha=0.1, color=color, label=label)
ax.legend(markerscale=5, fontsize=8)
style_axis(ax, title="PCA: PC1 vs PC2 (colored by Outcome)", xlabel="PC1", ylabel="PC2")

plt.tight_layout()
plt.show()

In [ ]:
X_cluster = X_scaled[:50000]

inertias = []
sil_scores = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=100)
    labels = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster, labels, sample_size=10000))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(K_range, inertias, marker="o", color=C["accent"], lw=2)
style_axis(ax, title="Elbow Plot", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax = axes[1]
ax.plot(K_range, sil_scores, marker="o", color=C["safe"], lw=2)
style_axis(ax, title="Silhouette Score", xlabel="Number of Clusters (k)", ylabel="Score")

plt.tight_layout()
plt.show()

In [ ]:
optimal_k = 5
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_scaled[:len(pca_df)])

pca_df_clustered = pca_df.copy()
pca_df_clustered["cluster"] = cluster_labels
pca_df_clustered["is_default"] = df.loc[pca_df.index, "is_default"].values
pca_df_clustered["grade"] = df.loc[pca_df.index, "grade"].values

profile = pca_df_clustered.groupby("cluster").agg(
    count=("loan_amnt", "count"),
    default_rate=("is_default", "mean"),
    avg_income=("annual_inc", "median"),
    avg_loan=("loan_amnt", "median"),
    avg_rate=("int_rate", "median"),
    avg_dti=("dti", "median"),
    avg_fico=("fico_range_low", "median"),
    avg_revol_util=("revol_util", "median"),
)
profile["pct_of_total"] = profile["count"] / profile["count"].sum()

print("Cluster Profiles:")
print(profile.round(2).to_string())

In [ ]:
cluster_grade = pd.crosstab(pca_df_clustered["cluster"], pca_df_clustered["grade"], normalize="index")
cluster_grade = cluster_grade[[c for c in grade_order if c in cluster_grade.columns]]

fig, ax = plt.subplots(figsize=(12, 6))
cluster_grade.plot(kind="bar", stacked=True, ax=ax,
                    color=[GRADE_COLORS.get(g, C["mid_gray"]) for g in cluster_grade.columns],
                    width=0.7)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(title="Grade", ncol=7, fontsize=9)
style_axis(ax, title="Grade Distribution by Cluster",
           subtitle="Which clusters contain the riskiest borrowers?",
           xlabel="Cluster", ylabel="% of Cluster")
plt.tight_layout()
plt.show()